In [14]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

In [15]:
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer

from sklearn.preprocessing import StandardScaler

In [16]:
x, y = load_breast_cancer(return_X_y=True)
x = StandardScaler().fit_transform(x)

In [17]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [18]:


class SoftmaxRegression:

    def __init__(self, n_classes, lr=0.1, n_iter=1000, reg=0.0):
        self.n_classes = n_classes
        self.lr        = lr
        self.n_iter    = n_iter
        self.reg       = reg
        self.W         = None   
        self.b         = None   
        self.loss_hist = []

    def _softmax(self, Z):
        Z_stable = Z - Z.max(axis=1, keepdims=True)
        E = np.exp(Z_stable)
        return E / E.sum(axis=1, keepdims=True)

    def _one_hot(self, y, K):
        N = len(y)
        Y = np.zeros((N, K))
        Y[np.arange(N), y] = 1
        return Y

    def fit(self, X, y):
        N, p = X.shape
        K    = self.n_classes

        self.W = np.random.randn(p, K) * 0.01
        self.b = np.zeros(K)

        Y = self._one_hot(y, K)     

        for i in range(self.n_iter):
            Z    = X @ self.W + self.b
            Phat = self._softmax(Z)   

            ce_loss = -np.mean(np.sum(Y * np.log(Phat + 1e-15), axis=1))
            reg_loss = (0.5 * self.reg) * np.sum(self.W ** 2)
            self.loss_hist.append(ce_loss + reg_loss)

            dZ = (Phat - Y) / N       
            dW = X.T @ dZ + self.reg * self.W  
            db = dZ.sum(axis=0)            

            self.W -= self.lr * dW
            self.b -= self.lr * db

        return self

    def predict_proba(self, X):
        Z = X @ self.W + self.b
        return self._softmax(Z)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

In [19]:
sft = SoftmaxRegression(n_classes=2, lr=0.5, n_iter=500)

In [20]:
sft.fit(x_train, y_train)

In [21]:
sft.score(x_test, y_test)

np.float64(0.9824561403508771)